### Import libraries

In [ ]:
import pandas as pd
import os
import seaborn as sns

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

### Data processing

In [ ]:
df_orders = pd.read_excel(io = r"../data/raw/Global Superstore.xlsx", sheet_name = "Orders")
df_returns = pd.read_excel(io = r"../data/raw/Global Superstore.xlsx", sheet_name = "Returns")
df_employees = pd.read_excel(io = r"../data/raw/Global Superstore.xlsx", sheet_name = "People")


duplicated_rows_count = df_orders.duplicated().sum()
df_orders = df_orders.drop_duplicates()

if duplicated_rows_count == 0:
    print("No duplicate rows were found")
else:
    print(f"{duplicated_rows_count} duplicate rows were found")

print("-----------------------------------------------\n\n")

df_orders["Delivery Time"] = (df_orders["Ship Date"] - df_orders["Order Date"]).dt.days
df_orders["Cost"] = df_orders["Sales"] - df_orders["Profit"]
df_orders["Original Price"] = df_orders["Sales"] / (1 - df_orders["Discount"])
df_orders["Product Cost"] = df_orders["Cost"] - df_orders["Shipping Cost"]
df_orders["Shipping Ratio"] = df_orders["Shipping Cost"] / df_orders["Sales"]
df_orders["Discount Value"] = df_orders["Original Price"] - df_orders["Sales"]
df_orders["Cost Ratio"] = df_orders["Cost"] / df_orders["Sales"] 
df_orders["Order Day"] = df_orders["Order Date"].dt.day
df_orders["Order Day Name"] = df_orders["Order Date"].dt.day_name()
df_orders["Is Weekend"] = False
df_orders.loc[(df_orders["Order Day Name"] == "Saturday") | (df_orders["Order Day Name"] == "Sunday"), "Is Weekend"] = True
df_orders["Order Month"] = df_orders["Order Date"].dt.month_name()    
df_orders["Order Year"] = df_orders["Order Date"].dt.year
df_orders["Profit Margin"] = round(df_orders["Profit"] / df_orders["Sales"], 2)


df_orders = df_orders[["Row ID", "Order ID", "Order Priority", "Order Date", "Order Day", "Order Month", "Order Year", "Order Day Name", "Is Weekend", "Ship Date", 
                         "Ship Mode", "Delivery Time", "Customer ID", "Customer Name", "Segment", "City", "State",
                         "Country", "Postal Code", "Market", "Region", "Product ID", "Category", "Sub-Category", "Product Name",
                         "Quantity", "Original Price", "Discount", "Discount Value", "Sales", "Profit", "Profit Margin", "Cost", "Shipping Cost", "Product Cost", "Shipping Ratio", "Cost Ratio"]]



df_orders["Order Priority"] = df_orders["Order Priority"].astype("category")
df_orders["Order Day"] = df_orders["Order Day"].astype("int8")
df_orders["Order Month"] = pd.Categorical(
                                            df_orders["Order Month"], 
                                            categories = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"],
                                            ordered = True
                                        )
df_orders["Order Year"] = df_orders["Order Year"].astype("int16")
df_orders["Order Day Name"] = pd.Categorical(
                                            df_orders["Order Day Name"], 
                                            categories = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
                                            ordered = True
                                        )
df_orders["Ship Mode"] = df_orders["Ship Mode"].astype("category")
df_orders["Delivery Time"] = df_orders["Delivery Time"].astype("int16")
df_orders["Segment"] = df_orders["Segment"].astype("category")
df_orders["Postal Code"] = df_orders["Postal Code"].astype("str")
df_orders["Category"] = df_orders["Category"].astype("category")
df_orders["Sub-Category"] = df_orders["Sub-Category"].astype("category")
df_orders["Quantity"] = df_orders["Quantity"].astype("int16")
df_orders["Market"] = df_orders["Market"].astype("category")
df_orders["Region"] = df_orders["Region"].astype("category")

df_orders.info()

print("\n-----------------------------------------------")
print("I optimized the dataset by converting categorical columns to the 'category' dtype and reducing numeric columns to smaller integer types where possible.\n" \
"This reduced memory usage and improved performance for groupby operations.\n")

df_united_states = df_orders.loc[df_orders["Country"] == "United States"]
print(f"Total number of null observations in the Postal Code column across all countries: {df_orders["Postal Code"].isnull().sum()}")
print(f"Total number of null observations in the Postal Code column for the United States: {df_united_states["Postal Code"].isnull().sum()}")
print("We can observe that the postal code is missing worldwide, except in the USA")
print("-----------------------------------------------\n\n")

df_orders["Postal Code"] = df_orders["Postal Code"].fillna("Unknown")

print("DATA AUDIT: Region Nomenclature Alignment\n"
      "ISSUE: 'AMEA' region detected in the Employees dataset, conflicting with 'EMEA' in the Orders dataset.\n"
      "VALIDATION: Filtering confirmed that staff assigned to European countries were incorrectly tagged as 'AMEA'.\n"
      "CONCLUSION: Identified 'AMEA' as a typographical error for 'EMEA' within the personnel records.\n"
      "ACTION: Standardized 'AMEA' to 'EMEA' to ensure consistent mapping between employees and sales data.\n\n")

print("List of all unique countries belonging to the EMEA region within the orders dataset. This confirms that the 'AMEA' region used in the employees dataset is a labeling error.\n")
print(f"{df_orders.loc[df_orders["Region"] == "EMEA"]["Country"].unique()}\n")

print("Employees dataframe")
df_employees["Region"] = df_employees["Region"].replace({"AMEA": "EMEA"})
print(df_employees.head(20))
print("-----------------------------------------------\n\n")



df_returns = df_returns[["Order ID", "Returned"]]
df_returns["Returned"] = True
df_employees["Region"] = df_employees["Region"].astype("category")

data_frame = pd.merge(left = df_orders, right = df_employees, how = "inner", on = "Region")
data_frame = pd.merge(left = data_frame, right = df_returns, how = "left", on = "Order ID")



print(f"Before merging with the returns dataframe: {df_orders.shape[0]} rows count")
print(f"After merging with the returns dataframe: {data_frame.shape[0]} rows count")
print(f"{data_frame.duplicated().sum()} duplicate rows were found\n")

data_frame = data_frame.drop_duplicates()

print("Due to inconsistencies in the dataset, some Order IDs were reused across different transactions. This caused unintended duplicates during the merge process.\n"
"After validating that these duplicates were identical and did not contain conflicting information, they were safely removed using drop_duplicates().")


data_frame["Returned"] = data_frame["Returned"].fillna(False)
data_frame["Returned"] = data_frame["Returned"].astype("bool")

data_frame = data_frame.rename(columns={"Person": "Employee"})
data_frame.columns = (data_frame.columns.str.lower().str.replace(" ", "_").str.replace("-", "_"))
data_frame = data_frame.reset_index(drop=True)


if os.path.isdir(r"../data/processed") is False:
    os.mkdir(r"../data/processed")

with pd.ExcelWriter(r"../data/processed/Global Superstore_processed.xlsx") as writer:
    data_frame.to_excel(writer, sheet_name = "Orders", header = True, index = False)



print("-----------------------------------------------\n\n")
print("The data has been processed!\n\n")
data_frame.info()
data_frame.head(50)

### Data analysis 

In [ ]:

total_rows = data_frame.shape[0]
total_columns = data_frame.shape[1]

total_days = len(data_frame["order_date"].unique())
total_orders = len(data_frame["order_id"].unique())
total_customers = len(data_frame["customer_id"].unique())
total_products = len(data_frame["product_id"].unique())
total_quantity_sold = data_frame["quantity"].sum()
total_countries = len(data_frame["country"].unique())

avg_orders_per_customer = round(total_orders / total_customers, 2)
avg_products_per_customer = round(total_quantity_sold / total_customers, 2)
avg_revenue_per_customer = round(data_frame["sales"].sum() / total_customers, 2)
avg_profit_per_customer = round(data_frame["profit"].sum() / total_customers, 2)

avg_products_per_order = round(total_quantity_sold / total_orders, 2)
avg_unique_products_per_order = round(data_frame.groupby("order_id")["product_id"].nunique().mean(), 2)
avg_revenue_per_order = round(data_frame["sales"].sum() / total_orders, 2)
avg_profit_per_order = round(data_frame["profit"].sum() / total_orders, 2)

profit_margin = round(data_frame["profit"].sum() / data_frame["sales"].sum(), 2)
return_rate = round(data_frame["returned"].mean(), 2)
profit_percentage = round((data_frame.loc[data_frame["profit"] > 0]["order_id"].count() / total_rows) * 100, 2)
loss_percentage = round((data_frame.loc[data_frame["profit"] <= 0]["order_id"].count() / total_rows) * 100, 2)

categorical_columns = [
    "order_priority",
    "order_day_name",
    "ship_mode",
    "segment",
    "country",
    "market",
    "category",
    "sub_category",
    "returned"
]


for column in categorical_columns:
    print(f"\n--------{column.upper()} DISTRIBUTION--------\n")
    
    df_dist = data_frame.groupby(column)["row_id"].count().sort_values(ascending=False).reset_index()
    df_dist = df_dist.rename(columns={"row_id": "count"})
    df_dist["%"] = round((df_dist["count"] / df_dist["count"].sum()) * 100, 2)
    
    print(df_dist.head(25))
    
    print("\nInsight:")
    print(f"- Most frequent value: {df_dist.iloc[0][column]} ({df_dist.iloc[0]['%']}%)")
    print(f"- Least frequent value: {df_dist.iloc[-1][column]} ({df_dist.iloc[-1]['%']}%)")
    print("------------------------------------------------------------\n")




print("\n--------PRODUCT PERFORMANCE--------\n")

print("Top 5 products by total revenue (Sales):")
print(data_frame.groupby("product_name")["sales"].sum().sort_values(ascending=False).head())

print("\nInsight:")
print("- These products bring the most money.")
print("- They are either popular or expensive products.\n")


print("Worst 5 products by total profit (loss-makers):")
print(data_frame.groupby("product_name")["profit"].sum().sort_values().head())

print("\nInsight:")
print("- These products lose money.")
print("- Possible reasons: big discounts, high costs, or low prices.\n")




print("\n--------RETURNS ANALYSIS--------\n")

print("Return rate by category:")
print(data_frame.groupby("category")["returned"].mean())

print("\nInsight:")
print("- Helps identify which product categories are more prone to returns.\n")

print("Return rate by shipping mode:")
print(data_frame.groupby("ship_mode")["returned"].mean())

print("\nInsight:")
print("- Can indicate if delivery speed impacts return behavior.\n")

print("Total returned items per category")
print(data_frame.groupby("category")["returned"].sum().reset_index().sort_values(by = "returned", ascending = False))
print("\n")

print("Total returned items per segment")
print(data_frame.groupby("segment")["returned"].sum().reset_index().sort_values(by = "returned", ascending = False))
print("\n")



print("\n--------PROFIT ANALYSIS--------\n")

print("Total profit by category:")
print(data_frame.groupby("category")["profit"].sum().reset_index().sort_values(by = "profit", ascending = False).head(10))
print("\nInsight:")
print("- Technology makes the most profit.")
print("- Office Supplies sells a lot but makes less profit per item.")
print("- Furniture makes the least profit.\n")

print("Total profit by market:")
print(data_frame.groupby("market")["profit"].sum().reset_index().sort_values(by = "profit", ascending = False).head(10))
print("\nInsight:")
print("- APAC and EU make the most profit.")
print("- Canada makes very little profit.\n")

print("Total profit by segment:")
print(data_frame.groupby("segment")["profit"].sum().reset_index().sort_values(by = "profit", ascending = False).head(10))
print("\n")
print("\nInsight:")
print("- Consumer segment brings most of the profit.")
print("- Home Office brings the least.\n")

print("Total profit by sub_category:")
print(data_frame.groupby("sub_category")["profit"].sum().reset_index().sort_values(by = "profit", ascending = False).head(10))
print("\nInsight:")
print("- Copiers and Phones make the most profit.")
print("- Products that sell a lot are not always the most profitable.")
print("- Expensive products bring more profit than cheap ones.\n")

print("Total profit by order_priority:")
print(data_frame.groupby("order_priority")["profit"].sum().reset_index().sort_values(by = "profit", ascending = False).head(10))
print("\n")

print("Total profit by country (top 10):")
print(data_frame.groupby("country")["profit"].sum().reset_index().sort_values(by = "profit", ascending = False).head(10))
print("\nInsight:")
print("- United States makes the most profit.")

print("Total profit by country (last 10):")
print(data_frame.groupby("country")["profit"].sum().reset_index().sort_values(by = "profit", ascending = True).head(10))
print("\nInsight:")
print("- Some countries lose money (like Turkey and Nigeria).")
print("- These markets may have pricing or cost problems.\n")

print("Relationship between profit and discount")
print(data_frame[["profit", "discount"]].corr())
print("\nInsight:")
print("- When discount increases, profit goes down.")
print("- Discounts reduce profit.\n")

print("Relationship between profit and shipping cost")
print(data_frame[["profit", "shipping_cost"]].corr())
print("\nInsight:")
print("- Higher shipping cost is linked to higher profit.")
print("- Probably expensive products cost more to ship.\n")

print("Relationship between profit and quantity")
print(data_frame[["profit", "quantity"]].corr())
print("\nInsight:")
print("- Selling more items does not increase profit much.")
print("- Profit depends more on price than quantity.\n")

print("Relationship between profit and total cost, product cost and shipping cost")
print(data_frame[["profit", "cost", "product_cost", "shipping_cost"]].corr())
print("\nInsight:")
print("- Cost and product cost are almost the same.")
print("- Shipping cost is an important part of total cost.")
print("- Cost does not strongly control profit.\n")






print("\n--------DISCOUNT & DELIVERY IMPACT--------\n")

print("Average discount for returned vs non-returned orders:")
print(data_frame.groupby("returned")["discount"].mean())
print("\nInsight:")
print("- Returned orders have smaller discounts.")
print("- Discounts are not the main reason for returns.\n")

print("Average delivery time for returned vs non-returned orders:")
print(data_frame.groupby("returned")["delivery_time"].mean())
print("\nInsight:")
print("- Delivery time is similar for returned and non-returned orders.")
print("- Shipping speed does not affect returns much.\n")

print(f"Average shipping cost per subcategories:")
print(data_frame.groupby("sub_category")["shipping_cost"].mean().reset_index().sort_values(by = "shipping_cost", ascending = False).head(25))
print("\nInsight:")
print("- Tables have the highest shipping cost.")
print("- Large and heavy products cost more to ship.")
print("- Furniture items (Tables, Bookcases) are expensive to deliver.")
print("- Electronics like Copiers and Phones also have high shipping cost.")

print("Relationship between sales and discount")
print(data_frame[["sales", "discount"]].corr())
print("\nInsight:")
print("- Discounts do not increase sales much.")
print("- Discount strategy is not very effective.\n")

print("Relationship between delivery time and shipping cost")
print(data_frame[["delivery_time", "shipping_cost"]].corr())
print("\nInsight:")
print("- Faster delivery costs more.")
print("- There is a trade-off between speed and cost.\n")

print("Relationship between profit sales and shipping ratio")
print(data_frame[["profit", "shipping_ratio"]].corr())
print("\nInsight:")
print("- Shipping ratio does not affect profit much.\n")

print("Relationship between quantity and shipping cost")
print(data_frame[["quantity", "shipping_cost"]].corr())
print("\nInsight:")
print("- Bigger orders have higher shipping cost.")
print("- Bulk shipping could be improved.\n")



print("\n--------KEY BUSINESS METRICS SUMMARY--------\n")

print(f"Dataset size: {total_rows} rows X {total_columns} columns")
print(f"Time span: {total_days} unique days")
print(f"Total orders: {total_orders}")
print(f"Tota countries: {total_countries}")
print(f"Profit distribution: {profit_percentage}% profit, {loss_percentage}% loss")


print("\nCustomer info:")
print(f"- Average orders per customer: {avg_orders_per_customer}")
print(f"- Average products per customer: {avg_products_per_customer}")
print(f"- Average revenue per customer: {avg_revenue_per_customer}")
print(f"- Average profit per customer: {avg_profit_per_customer}")

print("\nOrder info:")
print(f"- Average products per order: {avg_products_per_order}")
print(f"- Average unique products per order: {avg_unique_products_per_order}")
print(f"- Average revenue per order: {avg_revenue_per_order}")
print(f"- Average profit per order: {avg_profit_per_order}")
print(f"- Profit margin: {profit_margin}")

print("\nReturns:")
print(f"- Return rate: {return_rate}")

print("\nInventory:")
print(f"- Unique customers: {total_customers}")
print(f"- Unique products: {total_products}")
print(f"- Total quantity sold: {total_quantity_sold}")

print("\nInsight:")
print(f"- About 25% of orders lose money.")
print("- Discounts reduce profit but do not increase sales much.")
print("- Profit comes more from expensive products.")
print("- Some markets are not profitable.")
print("- Faster shipping costs more but does not reduce returns.")
print("- Consumer segment is the most important.\n")




print("\n--------NUMERICAL FEATURES SUMMARY--------\n")

print(data_frame[[
    "delivery_time", "quantity", "original_price", "discount",
    "discount_value", "sales", "profit", "profit_margin",
    "cost", "shipping_cost", "product_cost",
    "shipping_ratio", "cost_ratio"
]].describe())


print("\n--------DATASET INFO--------\n")
data_frame.info()

print("\n------------------------------------------------------------\n")

data_frame.head(50)

In [ ]:
pairplot_df = data_frame[[ "delivery_time", "quantity", "original_price", "discount",
    "discount_value", "sales", "profit", "profit_margin",
    "cost", "shipping_cost", "product_cost",
    "shipping_ratio", "cost_ratio"]]

print("Summary:")
print("- Profit is heavily impacted by discounts")
print("- Volume is not a primary profit driver")
print("- High-end products generate the majority of the revenue")
print("- There are outliers that can affect the analysis")
print("- Costs are highly correlated (data redundancy)")

sns.pairplot(data = pairplot_df)